# 09 — Contrôles entre tables Silver
**Objectif :** combler les trous de nettoyage : clés entre tables, valeurs autorisées, États, valeurs extrêmes.
**Ces contrôles ne modifient rien.** Ils comptent et signalent. Une anomalie trouvée se **marque** (indicateur) ou se **documente**.
Chaque contrôle est protégé : une erreur s'affiche en `❌ ERREUR` sans arrêter le notebook.

À lancer **après** les notebooks 02 à 08.

In [0]:
SILVER = "workspace.silver"


def check_sql(label, sql, attendu):
    """Exécute une requête qui renvoie un nombre, et le compare à l'attendu."""
    try:
        obtenu = spark.sql(sql).collect()[0][0]
        if hasattr(obtenu, "__float__") and not isinstance(obtenu, (int, bool)):
            obtenu = round(float(obtenu), 2)
        ok = obtenu == attendu
        print(("✅ OK     " if ok else "❌ ÉCART  ") + f"{label} : obtenu = {obtenu} | attendu = {attendu}")
    except Exception as e:
        print(f"❌ ERREUR  {label} : {str(e)[:150]}")

## 1. Clés entre tables (lignes orphelines : on attend 0)

In [0]:
S = SILVER
check_sql("orders -> customers", f"""SELECT COUNT(*) FROM {S}.silver_orders o
    LEFT JOIN {S}.silver_customers c ON o.customer_id = c.customer_id WHERE c.customer_id IS NULL""", 0)
check_sql("order_items -> orders", f"""SELECT COUNT(*) FROM {S}.silver_order_items i
    LEFT JOIN {S}.silver_orders o ON i.order_id = o.order_id WHERE o.order_id IS NULL""", 0)
check_sql("order_items -> products", f"""SELECT COUNT(*) FROM {S}.silver_order_items i
    LEFT JOIN {S}.silver_products p ON i.product_id = p.product_id WHERE p.product_id IS NULL""", 0)
check_sql("order_items -> sellers", f"""SELECT COUNT(*) FROM {S}.silver_order_items i
    LEFT JOIN {S}.silver_sellers s ON i.seller_id = s.seller_id WHERE s.seller_id IS NULL""", 0)
check_sql("order_payments -> orders", f"""SELECT COUNT(*) FROM {S}.silver_order_payments p
    LEFT JOIN {S}.silver_orders o ON p.order_id = o.order_id WHERE o.order_id IS NULL""", 0)
check_sql("order_reviews -> orders", f"""SELECT COUNT(*) FROM {S}.silver_order_reviews r
    LEFT JOIN {S}.silver_orders o ON r.order_id = o.order_id WHERE o.order_id IS NULL""", 0)

## 2. Cas connus du PDF (à marquer ou documenter)
- **R11** : commande avec articles mais **sans paiement** (attendu 1).
- **R10** : commandes **sans article** avec statut `created`, `invoiced` ou `shipped` (attendu 8).

In [0]:
check_sql("R11 : commandes avec articles mais sans paiement", f"""
    SELECT COUNT(DISTINCT i.order_id) FROM {S}.silver_order_items i
    LEFT JOIN {S}.silver_order_payments p ON i.order_id = p.order_id WHERE p.order_id IS NULL""", 1)

check_sql("R10 : sans article et statut created/invoiced/shipped", f"""
    SELECT COUNT(*) FROM {S}.silver_orders o
    LEFT JOIN (SELECT DISTINCT order_id FROM {S}.silver_order_items) i ON o.order_id = i.order_id
    WHERE i.order_id IS NULL AND o.order_status IN ('created','invoiced','shipped')""", 8)

check_sql("commandes sans article (au total)", f"""
    SELECT COUNT(*) FROM {S}.silver_orders o
    LEFT JOIN (SELECT DISTINCT order_id FROM {S}.silver_order_items) i ON o.order_id = i.order_id
    WHERE i.order_id IS NULL""", 775)

## 3. Valeurs autorisées

In [0]:
check_sql("statuts de commande distincts", f"SELECT COUNT(DISTINCT order_status) FROM {S}.silver_orders", 8)
check_sql("statuts hors liste", f"""SELECT COUNT(*) FROM {S}.silver_orders WHERE order_status NOT IN
    ('created','approved','invoiced','processing','shipped','delivered','unavailable','canceled')""", 0)
check_sql("types de paiement hors liste", f"""SELECT COUNT(*) FROM {S}.silver_order_payments WHERE payment_type NOT IN
    ('credit_card','boleto','voucher','debit_card','not_defined')""", 0)
check_sql("notes hors 1-5", f"""SELECT COUNT(*) FROM {S}.silver_order_reviews
    WHERE review_score IS NULL OR review_score NOT BETWEEN 1 AND 5""", 0)

## 4. États brésiliens valides (27)

In [0]:
ETATS = "'AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS','MG','PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR','SC','SP','SE','TO'"
check_sql("customers : État invalide", f"SELECT COUNT(*) FROM {S}.silver_customers WHERE customer_state NOT IN ({ETATS})", 0)
check_sql("sellers : État invalide", f"SELECT COUNT(*) FROM {S}.silver_sellers WHERE seller_state NOT IN ({ETATS})", 0)

## 5. Valeurs extrêmes (on regarde, on ne corrige pas)
Prix : minimum 0,85 et maximum 6 735 d'après ton PDF.

In [0]:
check_sql("prix minimum", f"SELECT MIN(price) FROM {S}.silver_order_items", 0.85)
check_sql("prix maximum", f"SELECT MAX(price) FROM {S}.silver_order_items", 6735.0)
check_sql("durée max achat -> livraison (jours, PDF : 210)", f"""
    SELECT MAX(DATEDIFF(order_delivered_ts, order_purchase_ts)) FROM {S}.silver_orders""", 210)

In [0]:
# Produits à regarder : poids à 0 ou dimensions absurdes (pas de valeur attendue, c'est une observation)
try:
    spark.sql(f"""
        SELECT SUM(CASE WHEN product_weight_g = 0 THEN 1 ELSE 0 END) AS poids_zero,
               MAX(product_weight_g) AS poids_max_g,
               MAX(product_length_cm) AS longueur_max_cm
        FROM {S}.silver_products
    """).show()
except Exception as e:
    print("Observation ignorée :", str(e)[:200])

## Résultat
Tout `❌ ÉCART` se traite ainsi : ouvrir la requête, comprendre pourquoi, puis **marquer ou documenter** (dans ton README, tableau « findings qualité »).
Une valeur qui diffère de ton PDF n'est pas forcément une erreur : c'est peut-être le PDF qui était approximatif.